# Study of the univariate distribution shift

## Experience 1 - Univariate CMIP distribution analysis from precomputed patch samples

This notebook analyzes the distribution of one CMIP variable using the precomputed multivariate patch samples.

Each sample is a geographic patch at one time step, containing `grid_points_per_patch` grid points and the full set of climate variables. The analysis aggregates the selected variable over all loaded samples and grid points, then compares the distributions across climates.

The notebook produces the following results :
- a heatmap of one selected sample;
- a latitude profile of the selected variable averaged over samples and longitude.
- raw distributions;
- normalized distributions;
- empirical CDFs;
- QQ-plots;
- Wasserstein and KS distances;
- moments;
- extreme quantiles.


## 0. Configuration

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Optional, only used for nicer display if available.
try:
    from IPython.display import display
except Exception:
    display = print

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

# --------------------------------------------------------------------------------------
# User parameters
# --------------------------------------------------------------------------------------

# Number of precomputed samples to load.
# This must correspond to an existing directory:
# /glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_v{num_samples}
num_samples = 100000

# Variable to analyze.
selected_variable = "wap500"

# Number of values sampled per climate for distribution diagnostics.
# Values are sampled from the loaded patch samples.
TARGET_POINTS_GLOBAL = 500_000

# Multi-seed resampling for uncertainty bands.
SEEDS = [7, 19, 31, 43, 59]

# Histogram bins.
HIST_BINS = 80

# Controls for the additional visual diagnostics.
sample_plot_climate = "historical"
sample_plot_index = 0

# Max number of samples used per climate for the latitude profile.
# Increase if you want a smoother profile and can afford the runtime.
PROFILE_MAX_SAMPLES_PER_CLIMATE = 100_000

# --------------------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------------------

precomputed_dir = Path(
    f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_samples}"
)

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

print("Precomputed directory:", precomputed_dir)


## 1. Load precomputed patch samples

In [ ]:
with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])

max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])

n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch

if selected_variable not in selected_variables_full:
    raise ValueError(
        f"{selected_variable!r} is not in selected_variables_full={selected_variables_full}"
    )

selected_var_idx = selected_variables_full.index(selected_variable)
selected_var_start = selected_var_idx * grid_points_per_patch
selected_var_end = selected_var_start + grid_points_per_patch

features_by_climate_full = {
    climate: np.load(precomputed_dir / f"features_{climate}.npy", mmap_mode="r")
    for climate in climate_order
}

metadata_by_climate = {
    climate: pd.read_csv(precomputed_dir / f"metadata_{climate}.csv")
    for climate in climate_order
}

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")

sample_pairs_by_climate = {
    climate: pd.read_csv(precomputed_dir / f"sample_pairs_{climate}.csv")
    for climate in climate_order
}

for climate, X in features_by_climate_full.items():
    if X.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {climate}: got {X.shape[1]}, expected {expected_dim_full}."
        )

print("Climate order:", climate_order)
print("Selected variables:", selected_variables_full)
print("Selected variable:", selected_variable)
print("Patch shape:", n_lat, "x", n_lon, "=", grid_points_per_patch)
print("Loaded feature shapes:")
for climate in climate_order:
    print(f"  {climate}: {features_by_climate_full[climate].shape}")

display(sample_count_df)
display(sampling_diagnostics_df)
display(patch_catalog.head())


## 2. Build variable value pools from samples

For each climate, we extract the selected variable from the precomputed samples.  
The feature layout is assumed to be variable-major:

```text
[var0_point0, ..., var0_point69, var1_point0, ..., var5_point69]
```

For a given variable, values are therefore located in a contiguous block of `grid_points_per_patch` columns.


In [ ]:
def get_variable_matrix(climate: str, variable: str) -> np.ndarray:
    """Return matrix (n_samples, grid_points_per_patch) for one variable and one climate."""
    var_idx = selected_variables_full.index(variable)
    start = var_idx * grid_points_per_patch
    end = start + grid_points_per_patch
    return np.asarray(features_by_climate_full[climate][:, start:end], dtype=np.float32)


def build_candidate_pool_from_samples(climate: str, variable: str) -> np.ndarray:
    """Flatten selected-variable values over all loaded samples and grid points."""
    values = get_variable_matrix(climate, variable).reshape(-1)
    values = values[np.isfinite(values)]
    return values.astype(float)


def sample_fixed_n(values: np.ndarray, n_points: int, rng: np.random.Generator) -> np.ndarray:
    """Sample n_points without replacement from a 1D array."""
    values = np.asarray(values)
    if n_points <= 0 or values.size == 0:
        return np.array([], dtype=float)
    n = min(int(n_points), values.size)
    idx = rng.choice(values.size, size=n, replace=False)
    return values[idx].astype(float)


def standardize(values: np.ndarray) -> np.ndarray:
    """Standardize a 1D array using its own mean/std."""
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    mu = np.nanmean(values)
    sigma = np.nanstd(values)
    if not np.isfinite(sigma) or sigma == 0:
        return values - mu
    return (values - mu) / sigma


def histogram_mean_std(values_by_seed, bins):
    """Mean and std of histogram densities across seeds."""
    densities = []
    for values in values_by_seed:
        hist, _ = np.histogram(values, bins=bins, density=True)
        densities.append(hist)
    densities = np.asarray(densities)
    return densities.mean(axis=0), densities.std(axis=0)


candidate_pool_by_climate = {
    climate: build_candidate_pool_from_samples(climate, selected_variable)
    for climate in climate_order
}

n_common_global = min(
    TARGET_POINTS_GLOBAL,
    min(candidate_pool_by_climate[climate].size for climate in climate_order),
)

if n_common_global == 0:
    raise ValueError("No finite values available for at least one climate.")

sampled_values_by_seed = {}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    sampled_values_by_seed[seed] = {
        climate: sample_fixed_n(candidate_pool_by_climate[climate], n_common_global, rng)
        for climate in climate_order
    }

sampled_values_by_climate = {
    climate: sampled_values_by_seed[SEEDS[0]][climate]
    for climate in climate_order
}

all_seed_values = np.concatenate([
    sampled_values_by_seed[seed][climate]
    for seed in SEEDS
    for climate in climate_order
])

candidate_summary_df = pd.DataFrame({
    "scenario": climate_order,
    "finite_values_available": [candidate_pool_by_climate[c].size for c in climate_order],
    "sampled_values_per_seed": [n_common_global] * len(climate_order),
}).set_index("scenario")

display(candidate_summary_df)
print(f"Selected variable: {selected_variable}")
print(f"Random seeds: {SEEDS}")
print(f"Common sampled values per climate and seed: n = {n_common_global:,}")


## 3. Additional spatial diagnostics

### 3.1 One sample heatmap

In [ ]:
def get_patch_info(patch_id: int) -> pd.Series:
    rows = patch_catalog.loc[patch_catalog["patch_id"] == patch_id]
    if rows.empty:
        raise ValueError(f"patch_id={patch_id} not found in patch_catalog.")
    return rows.iloc[0]


def get_patch_lat_lon_grids(patch_id: int):
    """Return local lon/lat grids for a patch using patch_catalog metadata."""
    patch_info = get_patch_info(int(patch_id))

    local_n_lat = int(patch_info["lat_stop_idx"] - patch_info["lat_start_idx"])
    local_n_lon = int(patch_info["lon_stop_idx"] - patch_info["lon_start_idx"])

    if local_n_lat * local_n_lon != grid_points_per_patch:
        raise ValueError(
            f"Patch shape mismatch for patch {patch_id}: "
            f"{local_n_lat} * {local_n_lon} != {grid_points_per_patch}"
        )

    lats = np.linspace(float(patch_info["lat_start"]), float(patch_info["lat_stop"]), local_n_lat)
    lons = np.linspace(float(patch_info["lon_start"]), float(patch_info["lon_stop"]), local_n_lon)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    return lat_grid, lon_grid, patch_info


import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature


def plot_sample_context_and_heatmap(climate: str, sample_index: int, variable: str):
    X_var = get_variable_matrix(climate, variable)
    metadata = metadata_by_climate[climate].reset_index(drop=True)

    if sample_index < 0 or sample_index >= X_var.shape[0]:
        raise IndexError(f"sample_index={sample_index} outside [0, {X_var.shape[0]})")

    patch_id = int(metadata.iloc[sample_index]["patch_id"])
    lat_grid, lon_grid, patch_info = get_patch_lat_lon_grids(patch_id)

    values = X_var[sample_index].reshape(n_lat, n_lon)

    lon_min = float(lon_grid.min())
    lon_max = float(lon_grid.max())
    lat_min = float(lat_grid.min())
    lat_max = float(lat_grid.max())

    fig = plt.figure(figsize=(13, 5.5), constrained_layout=True)

    # ------------------------------------------------------------
    # Left panel: global context map
    # ------------------------------------------------------------
    ax_world = fig.add_subplot(1, 2, 1, projection=ccrs.PlateCarree())

    ax_world.set_global()
    ax_world.coastlines(resolution="110m", linewidth=0.7, color="black")
    ax_world.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
    ax_world.gridlines(linewidth=0.3, alpha=0.35)

    rect = mpatches.Rectangle(
        (lon_min, lat_min),
        lon_max - lon_min,
        lat_max - lat_min,
        linewidth=2.0,
        edgecolor="red",
        facecolor="none",
        transform=ccrs.PlateCarree(),
        zorder=5,
    )

    ax_world.add_patch(rect)
    ax_world.set_title(f"Patch location on world map\npatch_id = {patch_id}")

    # ------------------------------------------------------------
    # Right panel: patch heatmap
    # ------------------------------------------------------------
    ax_patch = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())

    margin_lon = max(1.0, 0.2 * (lon_max - lon_min))
    margin_lat = max(1.0, 0.2 * (lat_max - lat_min))

    ax_patch.set_extent(
        [
            lon_min - margin_lon,
            lon_max + margin_lon,
            lat_min - margin_lat,
            lat_max + margin_lat,
        ],
        crs=ccrs.PlateCarree(),
    )

    ax_patch.coastlines(resolution="110m", linewidth=0.8, color="black")
    ax_patch.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")

    mesh = ax_patch.pcolormesh(
        lon_grid,
        lat_grid,
        values,
        shading="auto",
        cmap="viridis",
        transform=ccrs.PlateCarree(),
    )

    ax_patch.scatter(
        lon_grid.ravel(),
        lat_grid.ravel(),
        s=8,
        color="black",
        alpha=0.5,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )

    gl = ax_patch.gridlines(
        draw_labels=True,
        linewidth=0.3,
        alpha=0.35,
    )
    gl.top_labels = False
    gl.right_labels = False

    ax_patch.set_title(
        f"{variable} sample heatmap\n{climate}, sample {sample_index}"
    )

    fig.colorbar(
        mesh,
        ax=ax_patch,
        fraction=0.046,
        pad=0.04,
        label=variable,
    )

    display(metadata.iloc[[sample_index]])

    plt.show()


plot_sample_context_and_heatmap(
    sample_plot_climate,
    sample_plot_index,
    selected_variable,
)


### 3.2 Latitude profile

In [ ]:
def compute_latitude_profile(
    climate: str,
    variable: str,
    max_samples: int | None = PROFILE_MAX_SAMPLES_PER_CLIMATE,
    seed: int = random_seed,
) -> pd.DataFrame:
    """Average selected variable by latitude, over samples and longitude.

    The profile is computed from precomputed patch samples. For each selected sample,
    patch coordinates are recovered from patch_catalog. Values are then averaged by
    latitude across time/sample dimension and local longitude.
    """
    X_var = get_variable_matrix(climate, variable)
    metadata = metadata_by_climate[climate].reset_index(drop=True)

    n_samples_total = X_var.shape[0]
    if max_samples is not None and n_samples_total > max_samples:
        rng = np.random.default_rng(seed)
        sample_indices = np.sort(rng.choice(n_samples_total, size=max_samples, replace=False))
    else:
        sample_indices = np.arange(n_samples_total)

    X_sel = X_var[sample_indices]
    meta_sel = metadata.iloc[sample_indices].reset_index(drop=True)

    sum_by_lat = {}
    count_by_lat = {}
    lat_value_by_idx = {}

    for patch_id, local_rows in meta_sel.groupby("patch_id", sort=False).groups.items():
        local_idx = np.fromiter(local_rows, dtype=int)
        patch_id = int(patch_id)
        lat_grid, lon_grid, patch_info = get_patch_lat_lon_grids(patch_id)

        values = X_sel[local_idx].reshape(len(local_idx), n_lat, n_lon)

        lat_start_idx = int(patch_info["lat_start_idx"])
        for local_i in range(n_lat):
            global_lat_idx = lat_start_idx + local_i
            lat_value = float(lat_grid[local_i, 0])

            vals = values[:, local_i, :].reshape(-1)
            vals = vals[np.isfinite(vals)]

            if vals.size == 0:
                continue

            sum_by_lat[global_lat_idx] = sum_by_lat.get(global_lat_idx, 0.0) + float(vals.sum())
            count_by_lat[global_lat_idx] = count_by_lat.get(global_lat_idx, 0) + int(vals.size)
            lat_value_by_idx[global_lat_idx] = lat_value

    rows = []
    for global_lat_idx in sorted(sum_by_lat):
        rows.append({
            "scenario": climate,
            "global_lat_idx": global_lat_idx,
            "latitude": lat_value_by_idx[global_lat_idx],
            "mean_value": sum_by_lat[global_lat_idx] / count_by_lat[global_lat_idx],
            "n_values": count_by_lat[global_lat_idx],
        })

    return pd.DataFrame(rows)


latitude_profile_df = pd.concat(
    [
        compute_latitude_profile(climate, selected_variable, max_samples=PROFILE_MAX_SAMPLES_PER_CLIMATE)
        for climate in climate_order
    ],
    ignore_index=True,
)

display(latitude_profile_df.head())

fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)

for climate in climate_order:
    dfc = latitude_profile_df[latitude_profile_df["scenario"] == climate].sort_values("latitude")
    ax.plot(
        dfc["latitude"],
        dfc["mean_value"],
        linewidth=2.0,
        color=climate_colors[climate],
        label=climate,
    )

ax.set_title(f"Latitude profile of {selected_variable} averaged over samples and longitude")
ax.set_xlabel("Latitude")
ax.set_ylabel(f"Mean {selected_variable}")
ax.grid(alpha=0.25)
ax.legend(title="Climate")
plt.show()


## 4. Raw distributions

In [ ]:
global_bins = np.linspace(all_seed_values.min(), all_seed_values.max(), HIST_BINS + 1)
bin_centers = 0.5 * (global_bins[:-1] + global_bins[1:])

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True, sharex=True, sharey=True)
axes = axes.ravel()

for ax, climate in zip(axes, climate_order):
    values_by_seed = [sampled_values_by_seed[seed][climate] for seed in SEEDS]
    mean_density, std_density = histogram_mean_std(values_by_seed, global_bins)

    ax.plot(
        bin_centers,
        mean_density,
        color=climate_colors[climate],
        linewidth=2.0,
        label="Mean density",
    )
    ax.fill_between(
        bin_centers,
        np.maximum(mean_density - std_density, 0),
        mean_density + std_density,
        color=climate_colors[climate],
        alpha=0.22,
        label="±1 std across seeds",
    )

    ax.set_title(climate)
    ax.set_xlabel(selected_variable)
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)

axes[0].legend(loc="upper right")
fig.suptitle(f"Distribution of {selected_variable} across climates", fontsize=14)
plt.show()


fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

for climate in climate_order:
    values_by_seed = [sampled_values_by_seed[seed][climate] for seed in SEEDS]
    mean_density, std_density = histogram_mean_std(values_by_seed, global_bins)

    ax.plot(
        bin_centers,
        mean_density,
        linewidth=1.8,
        color=climate_colors[climate],
        label=climate,
    )
    ax.fill_between(
        bin_centers,
        np.maximum(mean_density - std_density, 0),
        mean_density + std_density,
        color=climate_colors[climate],
        alpha=0.15,
    )

ax.set_title(f"{selected_variable} — overlaid distributions across climates")
ax.set_xlabel(selected_variable)
ax.set_ylabel("Density")
ax.legend(title="Climate")
ax.grid(alpha=0.25)
plt.show()


## 5. Normalized distributions

In [ ]:
std_bins = np.linspace(-6, 6, HIST_BINS + 1)
std_centers = 0.5 * (std_bins[:-1] + std_bins[1:])

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True, sharex=True, sharey=True)
axes = axes.ravel()

for ax, climate in zip(axes, climate_order):
    standardized_by_seed = [
        standardize(sampled_values_by_seed[seed][climate])
        for seed in SEEDS
    ]
    mean_density, std_density = histogram_mean_std(standardized_by_seed, std_bins)

    ax.plot(
        std_centers,
        mean_density,
        color=climate_colors[climate],
        linewidth=2.0,
        label="Mean density",
    )
    ax.fill_between(
        std_centers,
        np.maximum(mean_density - std_density, 0),
        mean_density + std_density,
        color=climate_colors[climate],
        alpha=0.22,
        label="±1 std across seeds",
    )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{climate} standardized")
    ax.set_xlabel(f"{selected_variable} standardized")
    ax.set_ylabel("Density")
    ax.grid(alpha=0.2)

axes[0].legend(loc="upper right")
fig.suptitle(f"Standardized distribution of {selected_variable} across climates", fontsize=14)
plt.show()


fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

for climate in climate_order:
    standardized_by_seed = [
        standardize(sampled_values_by_seed[seed][climate])
        for seed in SEEDS
    ]
    mean_density, std_density = histogram_mean_std(standardized_by_seed, std_bins)

    ax.plot(
        std_centers,
        mean_density,
        linewidth=1.8,
        color=climate_colors[climate],
        label=climate,
    )
    ax.fill_between(
        std_centers,
        np.maximum(mean_density - std_density, 0),
        mean_density + std_density,
        color=climate_colors[climate],
        alpha=0.15,
    )

ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_title(f"{selected_variable} — overlaid standardized distributions")
ax.set_xlabel(f"{selected_variable} standardized")
ax.set_ylabel("Density")
ax.legend(title="Climate")
ax.grid(alpha=0.25)
plt.show()


## 6. Empirical CDFs

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)

x_grid = np.linspace(all_seed_values.min(), all_seed_values.max(), 500)

for climate in climate_order:
    cdf_stack = []
    for seed in SEEDS:
        v = np.sort(sampled_values_by_seed[seed][climate])
        cdf = np.searchsorted(v, x_grid, side="right") / v.size
        cdf_stack.append(cdf)

    cdf_stack = np.asarray(cdf_stack)
    cdf_mean = cdf_stack.mean(axis=0)
    cdf_std = cdf_stack.std(axis=0)

    ax.plot(
        x_grid,
        cdf_mean,
        linewidth=2,
        color=climate_colors[climate],
        label=climate,
    )
    ax.fill_between(
        x_grid,
        np.clip(cdf_mean - cdf_std, 0, 1),
        np.clip(cdf_mean + cdf_std, 0, 1),
        color=climate_colors[climate],
        alpha=0.15,
    )

ax.set_title(f"{selected_variable} — empirical CDF")
ax.set_xlabel(selected_variable)
ax.set_ylabel("F(x)")
ax.legend(title="Climate")
ax.grid(alpha=0.25)
plt.show()


## 7. QQ-plots vs historical

In [ ]:
def qq_data(reference, target, n_quantiles=500):
    probs = np.linspace(0.001, 0.999, n_quantiles)
    ref_q = np.quantile(reference, probs)
    target_q = np.quantile(target, probs)
    return ref_q, target_q


ssp_scenarios = [climate for climate in climate_order if climate != "historical"]

fig, axes = plt.subplots(1, len(ssp_scenarios), figsize=(16, 5), constrained_layout=True)

if len(ssp_scenarios) == 1:
    axes = [axes]

for ax, climate in zip(axes, ssp_scenarios):
    ref_q_stack = []
    target_q_stack = []

    for seed in SEEDS:
        ref_q, target_q = qq_data(
            sampled_values_by_seed[seed]["historical"],
            sampled_values_by_seed[seed][climate],
            n_quantiles=300,
        )
        ref_q_stack.append(ref_q)
        target_q_stack.append(target_q)

    ref_q_stack = np.asarray(ref_q_stack)
    target_q_stack = np.asarray(target_q_stack)

    ref_q_mean = ref_q_stack.mean(axis=0)
    target_q_mean = target_q_stack.mean(axis=0)
    target_q_std = target_q_stack.std(axis=0)

    ax.plot(ref_q_mean, target_q_mean, color=climate_colors[climate], linewidth=2)
    ax.fill_between(
        ref_q_mean,
        target_q_mean - target_q_std,
        target_q_mean + target_q_std,
        color=climate_colors[climate],
        alpha=0.2,
    )

    line_min = min(ref_q_mean.min(), target_q_mean.min())
    line_max = max(ref_q_mean.max(), target_q_mean.max())

    ax.plot(
        [line_min, line_max],
        [line_min, line_max],
        linestyle="--",
        color="black",
        linewidth=1.5,
    )

    ax.set_title(f"QQ-plot: historical vs {climate}")
    ax.set_xlabel("Quantiles historical")
    ax.set_ylabel(f"Quantiles {climate}")
    ax.grid(alpha=0.25)

plt.show()


## 8. Wasserstein and KS distances

In [ ]:
shift_rows = []

for seed in SEEDS:
    hist_values = sampled_values_by_seed[seed]["historical"]

    for climate in ssp_scenarios:
        target = sampled_values_by_seed[seed][climate]

        wasserstein = stats.wasserstein_distance(hist_values, target)
        ks_stat, ks_pvalue = stats.ks_2samp(
            hist_values,
            target,
            alternative="two-sided",
            method="auto",
        )

        shift_rows.append({
            "seed": seed,
            "scenario": climate,
            "wasserstein": wasserstein,
            "ks_stat": ks_stat,
            "ks_pvalue": ks_pvalue,
        })

shift_seed_df = pd.DataFrame(shift_rows)

shift_metrics_df = (
    shift_seed_df.groupby("scenario")[["wasserstein", "ks_stat", "ks_pvalue"]]
    .agg(["mean", "std"])
    .loc[ssp_scenarios]
)

display(
    shift_metrics_df.style.format({
        ("wasserstein", "mean"): "{:.6g}",
        ("wasserstein", "std"): "{:.2g}",
        ("ks_stat", "mean"): "{:.6g}",
        ("ks_stat", "std"): "{:.2g}",
        ("ks_pvalue", "mean"): "{:.3e}",
        ("ks_pvalue", "std"): "{:.2e}",
    })
)

plot_order = list(shift_metrics_df.index)
wass_mean = shift_metrics_df[("wasserstein", "mean")]
wass_std = shift_metrics_df[("wasserstein", "std")]
ks_mean = shift_metrics_df[("ks_stat", "mean")]
ks_std = shift_metrics_df[("ks_stat", "std")]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].bar(
    plot_order,
    wass_mean.values,
    yerr=wass_std.values,
    color=[climate_colors[c] for c in plot_order],
    capsize=4,
    alpha=0.9,
)
axes[0].set_title("Wasserstein distance vs historical")
axes[0].set_ylabel("Distance")
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(
    plot_order,
    ks_mean.values,
    yerr=ks_std.values,
    color=[climate_colors[c] for c in plot_order],
    capsize=4,
    alpha=0.9,
)
axes[1].set_title("Kolmogorov-Smirnov statistic vs historical")
axes[1].set_ylabel("KS statistic")
axes[1].grid(axis="y", alpha=0.25)

plt.show()


In [ ]:
pairwise_climates = climate_order


def pairwise_seed_mean_matrix(distance_fn):
    matrix = np.zeros((len(pairwise_climates), len(pairwise_climates)), dtype=float)

    for i, climate_i in enumerate(pairwise_climates):
        for j in range(i + 1, len(pairwise_climates)):
            climate_j = pairwise_climates[j]

            per_seed_distances = [
                distance_fn(
                    sampled_values_by_seed[seed][climate_i],
                    sampled_values_by_seed[seed][climate_j],
                )
                for seed in SEEDS
            ]

            mean_distance = float(np.mean(per_seed_distances))
            matrix[i, j] = mean_distance
            matrix[j, i] = mean_distance

    return matrix


wasserstein_matrix = pairwise_seed_mean_matrix(
    lambda values_a, values_b: stats.wasserstein_distance(values_a, values_b)
)

ks_matrix = pairwise_seed_mean_matrix(
    lambda values_a, values_b: stats.ks_2samp(
        values_a,
        values_b,
        alternative="two-sided",
        method="auto",
    ).statistic
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

heatmaps = [
    (wasserstein_matrix, "Wasserstein distance", "viridis"),
    (ks_matrix, "KS statistic", "magma"),
]

for ax, (matrix, title, cmap) in zip(axes, heatmaps):
    im = ax.imshow(matrix, cmap=cmap, vmin=0)

    ax.set_xticks(range(len(pairwise_climates)))
    ax.set_xticklabels(pairwise_climates, rotation=45, ha="right")
    ax.set_yticks(range(len(pairwise_climates)))
    ax.set_yticklabels(pairwise_climates)
    ax.set_title(title)
    ax.set_xlabel("Climate")
    ax.set_ylabel("Climate")

    max_val = matrix.max() if matrix.max() > 0 else 1.0

    for i in range(len(pairwise_climates)):
        for j in range(len(pairwise_climates)):
            ax.text(
                j,
                i,
                f"{matrix[i, j]:.3g}",
                ha="center",
                va="center",
                color="white" if matrix[i, j] > 0.5 * max_val else "black",
                fontsize=9,
            )

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()


## 9. Moments

In [ ]:
def summarize_moments(values):
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=0)),
        "skew": float(stats.skew(values, bias=False, nan_policy="omit")),
        "kurtosis": float(stats.kurtosis(values, fisher=True, bias=False, nan_policy="omit")),
    }


moment_rows = []

for seed in SEEDS:
    for climate in climate_order:
        values = sampled_values_by_seed[seed][climate]
        row = {"seed": seed, "scenario": climate}
        row.update(summarize_moments(values))
        moment_rows.append(row)

moments_seed_df = pd.DataFrame(moment_rows)

moments_df = (
    moments_seed_df.groupby("scenario")[["mean", "std", "skew", "kurtosis"]]
    .mean()
    .loc[climate_order]
)

moments_std_df = (
    moments_seed_df.groupby("scenario")[["mean", "std", "skew", "kurtosis"]]
    .std()
    .loc[climate_order]
)

moments_delta_df = moments_df.subtract(moments_df.loc["historical"], axis=1)

print("Raw moments (mean across seeds)")
display(moments_df.style.format("{:.6g}"))

print("Uncertainty (std across seeds)")
display(moments_std_df.style.format("{:.2g}"))

print("Deviation from historical")
display(moments_delta_df.style.format("{:+.6g}"))

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

for ax, metric in zip(axes.ravel(), ["mean", "std", "skew", "kurtosis"]):
    x = np.arange(len(climate_order))
    y = moments_df[metric].values
    yerr = moments_std_df[metric].values

    ax.bar(
        x,
        y,
        yerr=yerr,
        capsize=4,
        color=[climate_colors[c] for c in climate_order],
        alpha=0.9,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(climate_order, rotation=30, ha="right")
    ax.set_title(f"{metric}")
    ax.grid(axis="y", alpha=0.25)

plt.show()


In [ ]:
moment_shift_matrix = moments_delta_df.loc[
    climate_order,
    ["mean", "std", "skew", "kurtosis"],
]

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

max_abs_shift = np.nanmax(np.abs(moment_shift_matrix.values))
if max_abs_shift == 0 or not np.isfinite(max_abs_shift):
    max_abs_shift = 1.0

im = ax.imshow(
    moment_shift_matrix.values,
    cmap="coolwarm",
    vmin=-max_abs_shift,
    vmax=max_abs_shift,
    aspect="auto",
)

ax.set_xticks(range(moment_shift_matrix.shape[1]))
ax.set_xticklabels(moment_shift_matrix.columns, rotation=30, ha="right")
ax.set_yticks(range(moment_shift_matrix.shape[0]))
ax.set_yticklabels(moment_shift_matrix.index)
ax.set_title("Shift of moments vs historical")
ax.set_xlabel("Moment")
ax.set_ylabel("Climate")

for i in range(moment_shift_matrix.shape[0]):
    for j in range(moment_shift_matrix.shape[1]):
        value = moment_shift_matrix.values[i, j]
        ax.text(
            j,
            i,
            f"{value:+.3g}",
            ha="center",
            va="center",
            color="white" if abs(value) > 0.5 * max_abs_shift else "black",
            fontsize=9,
        )

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Shift vs historical")
plt.show()


## 10. Extreme quantiles

In [ ]:
quantile_rows = []

for seed in SEEDS:
    for climate in climate_order:
        values = sampled_values_by_seed[seed][climate]

        quantile_rows.append({
            "seed": seed,
            "scenario": climate,
            "q95": float(np.quantile(values, 0.95)),
            "q99": float(np.quantile(values, 0.99)),
            "max": float(np.max(values)),
        })

extremes_seed_df = pd.DataFrame(quantile_rows)

extremes_df = (
    extremes_seed_df.groupby("scenario")[["q95", "q99", "max"]]
    .mean()
    .loc[climate_order]
)

extremes_std_df = (
    extremes_seed_df.groupby("scenario")[["q95", "q99", "max"]]
    .std()
    .loc[climate_order]
)

extremes_delta_df = extremes_df.subtract(extremes_df.loc["historical"], axis=1)

print("Extreme quantiles (mean across seeds)")
display(extremes_df.style.format("{:.6g}"))

print("Uncertainty (std across seeds)")
display(extremes_std_df.style.format("{:.2g}"))

print("Differences relative to historical")
display(extremes_delta_df.style.format("{:+.6g}"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
x = np.arange(len(climate_order))

axes[0].bar(
    x - 0.17,
    extremes_df["q95"].values,
    width=0.34,
    yerr=extremes_std_df["q95"].values,
    capsize=4,
    color=[climate_colors[c] for c in climate_order],
    alpha=0.9,
    label="q95",
)

axes[0].bar(
    x + 0.17,
    extremes_df["q99"].values,
    width=0.34,
    yerr=extremes_std_df["q99"].values,
    capsize=4,
    color=[climate_colors[c] for c in climate_order],
    alpha=0.55,
    label="q99",
)

axes[0].set_xticks(x)
axes[0].set_xticklabels(climate_order, rotation=30, ha="right")
axes[0].set_title("95th and 99th percentiles")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend()

axes[1].bar(
    x - 0.17,
    extremes_delta_df["q95"].values,
    width=0.34,
    color=[climate_colors[c] for c in climate_order],
    alpha=0.9,
    label="q95 delta",
)

axes[1].bar(
    x + 0.17,
    extremes_delta_df["q99"].values,
    width=0.34,
    color=[climate_colors[c] for c in climate_order],
    alpha=0.55,
    label="q99 delta",
)

axes[1].set_xticks(x)
axes[1].set_xticklabels(climate_order, rotation=30, ha="right")
axes[1].set_title("Shift of extremes vs historical")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend()

plt.show()
